In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime 
import duckdb
import unicodedata
import sys
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

[04/21/26 12:19:49] INFO     Using                                                                  ]8;id=481901;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\framework\project\__init__.py\__init__.py]8;;\:]8;id=720892;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\framework\project\__init__.py#269\269]8;;\
                             'c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\framewo                
                             rk\project\rich_logging.yml' as logging configuration.                                

[04/21/26 12:19:50] WARNING  c:\Users\User\miniconda3\envs\central\Lib\site-packages\requests\__ini ]8;id=470610;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py\warnings.py]8;;\:]8;id=602120;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py#110\110]8;;\
                             t__.py:113: RequestsDependencyWarning: urllib3 (2.6.1) or chardet                     
                             (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported                    
                             version!                                                                              
                               warnings.warn(                                                                      
                                                                                                                   

                    INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=548833;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=685506;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

✅ Kedro context loaded! Project root: g:\Unidades compartidas\Alianzas\3. Data\CENTRAL\central-perm-flow


In [2]:
# Add the 'src' directory to the path
sys.path.append(os.path.abspath("src"))
import central_perm_flow.pipelines.data_processing.nodes as nodes_dproc
import central_perm_flow.pipelines.calac_activos_baj_grad.nodes as nodes_abg
import central_perm_flow.pipelines.cascadas_lifetables_survival.nodes as nodes_surv

# Fuentes de información

In [3]:
central_calendario_extendido_uptoday = catalog.load('central_calendario_extendido_uptoday')
central_bajas_calendario_academico = catalog.load('central_bajas_calendario_academico')
central_graduados_calendario_academico = catalog.load('central_graduados_calendario_academico')
central_activos_calendario = catalog.load('central_activos_calendario')
central_estados_calac = catalog.load('central_estados_calac')


[04/21/26 12:19:51] INFO     Loading data from central_calendario_extendido_uptoday            ]8;id=833651;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=795306;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

                    INFO     Loading data from central_bajas_calendario_academico              ]8;id=981885;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=191678;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

                    INFO     Loading data from central_graduados_calendario_academico          ]8;id=463268;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=660019;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

                    INFO     Loading data from central_activos_calendario (ParquetDataset)...  ]8;id=596546;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=925613;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

                    INFO     Loading data from central_estados_calac (ExcelDataset)...         ]8;id=476547;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=325478;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

# Semanal

## Parámetros

In [4]:
parameters = catalog.load('parameters')
dict_niveles_duracion = parameters['graduados_calac']['dict_niveles_duracion']

[04/21/26 12:19:52] INFO     Loading data from parameters (MemoryDataset)...                   ]8;id=720814;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=560993;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [5]:
# parámetros 
columnas_agrupacion = [
    'cohorte_inicial', 'fecha_ingreso', 'nivel_academico', 'nivel', 
    'programa', 'fecha_inicio', 'fecha_fin', 'semana', 
    'semana_acumulada', 'month', 'mes_academico', 'student_journey'
]

columnas_to_order = ['fecha_ingreso','month','fecha_inicio', 'fecha_fin']

group_columnas_agrupacion = [
    'cohorte_inicial', 'fecha_ingreso', 'nivel_academico', 'nivel', 'programa'
]

columnas_tokeep = ['cohorte_inicial',
                    'fecha_ingreso', 
                    'nivel_academico', 
                    'nivel',
                    'programa',  
                    'fecha_inicio', 
                    'fecha_fin', 
                    'semana',
                    'semana_acumulada',
                    'semana_limite',  
                    'month', 
                    'mes_academico', 
                    'nuevos', 
                    'ai',
                    'di',
                    'gi', 
                    'engi',
                    'ci' ]


survival_columns = ['cohorte_inicial',
                    'fecha_ingreso',
                    'nivel_academico',
                    'nivel',
                    'programa',
                    'fecha_inicio',
                    'fecha_fin', 
                    'semana',
                    'semana_acumulada',
                    'semana_limite',
                    'month',
                    'mes_academico',
                    'student_journey',
                    'nuevos',
                    'di',
                    'gi',
                    'engi', 
                    'ci',
                    'di_cum',
                    'gi_cum', 
                    'ai', 
                    'ni']



# Lifetables
- Cascadas 
- Sobrevivencia

## Semanal

In [6]:
cascadas_semanal_podada_censuras = catalog.load('cascadas_semanal_podada_censuras')

                    INFO     Loading data from cascadas_semanal_podada_censuras                ]8;id=559005;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=209094;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ExcelDataset)...                                                                     

In [7]:
group_cols = ['cohorte_inicial', 'fecha_ingreso', 'nivel_academico', 'nivel',
                    'programa', ]

unidades_tiempo =  ['fecha_inicio', 'fecha_fin', 'semana_acumulada']

# tabla de vida
central_tabla_vida_semanal = nodes_surv.calcular_km_y_eti_dinamico(
    df = cascadas_semanal_podada_censuras , 
    unidades_tiempo=unidades_tiempo,
    group_cols = group_cols
)


In [ ]:
mask_cohorte = central_tabla_vida_semanal['cohorte_inicial'] == '2025 1a'
mask_cohorte = central_tabla_vida_semanal['nivel_academico'] == 'posgrado'
mask_cohorte = central_tabla_vida_semanal['programa'] == 'auditoria y control'



0       True
1       True
2       True
3       True
4       True
       ...  
296    False
297    False
298    False
299    False
300    False
Name: programa, Length: 301, dtype: bool

In [ ]:
central_tabla_vida_semanal['cohorte_inicial']

,cohorte_inicial,fecha_ingreso,nivel_academico,nivel,programa,fecha_inicio,fecha_fin,semana_acumulada,nuevos,n_total,...,engi,ci,ci_cum,qi,pi,km,km_se,km_ic_inf,km_ic_sup,eti
0,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,2025-03-17,2025-03-17,0,29,29,...,0,0,0,0.000000,1.000000,1.000000,0.000000,1.000000,1.0,1.000000
1,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,2025-03-17,2025-03-23,1,29,29,...,0,0,0,0.000000,1.000000,1.000000,0.000000,1.000000,1.0,1.000000
2,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,2025-03-24,2025-03-30,2,29,29,...,0,0,0,0.000000,1.000000,1.000000,0.000000,1.000000,1.0,1.000000
3,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,2025-03-31,2025-04-06,3,29,29,...,0,0,0,0.068966,0.931034,0.931034,0.047054,0.838808,1.0,0.931034
4,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,2025-04-07,2025-04-13,4,29,29,...,0,0,0,0.000000,1.000000,0.931034,0.047054,0.838808,1.0,0.931034
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
981,2026 1b,2026-03-16,pregrado,pregrado,economia y finanzas,2026-03-23,2026-03-29,2,31,31,...,0,0,0,0.000000,1.000000,1.000000,0.000000,1.000000,1.0,1.000000
982,2026 1b,2026-03-16,pregrado,pregrado,economia y finanzas,2026-03-30,2026-04-05,3,31,31,...,0,0,0,0.000000,1.000000,1.000000,0.000000,1.000000,1.0,1.000000
983,2026 1b,2026-03-16,pregrado,pregrado,economia y finanzas,2026-04-06,2026-04-12,4,31,31,...,0,0,0,0.000000,1.000000,1.000000,0.000000,1.000000,1.0,1.000000
984,2026 1b,2026-03-16,pregrado,pregrado,economia y finanzas,2026-04-13,2026-04-19,5,31,31,...,0,0,0,0.000000,1.000000,1.000000,0.000000,1.000000,1.0,1.000000


# Mensual

### Parámetros

In [9]:
group_columns = [
                'cohorte_inicial', 
                'fecha_ingreso',
                'nivel_academico', 
                'nivel',
                'programa',  
                'month',
                'mes_academico'
                    ]

In [11]:
cascadas_mensual_podada_censuras = cascadas_semanal_podada_censuras.groupby(group_columns).agg({
                                                            'fecha_inicio':'min',
                                                            'fecha_fin':'max',
                                                            'nuevos': 'max',
                                                            'ai':'min',
                                                            'di':'sum',
                                                            'gi':'sum',
                                                            'engi':'sum',
                                                            'ci':'sum'}).reset_index()

In [14]:
group_cols = ['cohorte_inicial', 'fecha_ingreso', 'nivel_academico', 'nivel',
                    'programa', ]

unidades_tiempo =  ['fecha_inicio', 'fecha_fin', 'month']

# tabla de vida
central_tabla_vida_mensual = nodes_surv.calcular_km_y_eti_dinamico(
    df = cascadas_mensual_podada_censuras , 
    unidades_tiempo=unidades_tiempo,
    group_cols = group_cols
)


In [15]:
central_tabla_vida_mensual

,cohorte_inicial,fecha_ingreso,nivel_academico,nivel,programa,fecha_inicio,fecha_fin,month,nuevos,n_total,...,engi,ci,ci_cum,qi,pi,km,km_se,km_ic_inf,km_ic_sup,eti
0,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,2025-03-17,2025-03-17,0,29,29,...,0,0,0,0.000000,1.000000,1.000000,0.000000,1.000000,1.0,1.000000
1,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,2025-03-17,2025-04-13,1,29,29,...,0,0,0,0.068966,0.931034,0.931034,0.047054,0.838808,1.0,0.931034
2,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,2025-04-14,2025-05-11,2,29,29,...,0,0,0,0.000000,1.000000,0.931034,0.047054,0.838808,1.0,0.931034
3,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,2025-05-12,2025-06-08,3,29,29,...,0,0,0,0.037037,0.962963,0.896552,0.056552,0.785709,1.0,0.896552
4,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,2025-06-09,2025-07-06,4,29,29,...,0,0,0,0.000000,1.000000,0.896552,0.056552,0.785709,1.0,0.896552
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296,2026 1b,2026-03-16,pregrado,pregrado,derecho,2026-03-16,2026-04-12,1,75,75,...,0,0,0,0.013333,0.986667,0.986667,0.013244,0.960708,1.0,0.986667
297,2026 1b,2026-03-16,pregrado,pregrado,derecho,2026-04-13,2026-04-26,2,75,1,...,0,74,74,0.000000,1.000000,0.986667,0.013244,0.960708,1.0,0.986667
298,2026 1b,2026-03-16,pregrado,pregrado,economia y finanzas,2026-03-16,2026-03-16,0,31,31,...,0,0,0,0.000000,1.000000,1.000000,0.000000,1.000000,1.0,1.000000
299,2026 1b,2026-03-16,pregrado,pregrado,economia y finanzas,2026-03-16,2026-04-12,1,31,31,...,0,0,0,0.000000,1.000000,1.000000,0.000000,1.000000,1.0,1.000000
